In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from moabb.paradigms import MotorImagery
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

datasets = [
    AlexMI(),
    #BNCI2014_001(),
    #PhysionetMI(),
    #Schirrmeister2017(),
    #Weibo2014(),
    #Zhou2016()
]

n_classes=3
sfreq=250

In [3]:
import copy
from sklearn.base import clone
import dask
import os
from sklearn.preprocessing import FunctionTransformer
import tensorly as tl

cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

def eval_moabb_within_session(dataset, subject, pipe):
    subj_dataset = copy.deepcopy(dataset)

    events = list(subj_dataset.event_id.keys())[:n_classes]
    #paradigm = MotorImagery(events=events, n_classes=n_classes, resample=sfreq)
    paradigm = MotorImagery(resample=sfreq)

    subj_dataset = copy.deepcopy(dataset)
    n_subjects = len(dataset.subject_list)
    subj_dataset.subject_list = [subject]
    
    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=subj_dataset,
        overwrite=True,
        random_state=42,
        n_jobs=-1,
        suffix=f'bttda_dask_dataset-{dataset.code}_subject-{subject}_pipe-{pipe}',
        cache_config=cache_config,
    )
    print(f'dataset={dataset.code}, subject={subject}/{n_subjects}, pipe={pipe}')
    return evaluation.process({pipe:clone(pipelines[pipe])})





In [4]:
from classification_mi import get_pipelines_mi
pipelines = get_pipelines_mi()
pipelines


{'HODA': Pipeline(steps=[('stf',
                  FunctionTransformer(func=<function stf_transform at 0x1482c00bcea0>)),
                 ('tensorly',
                  FunctionTransformer(func=<function NumpyBackend.tensor at 0x14830690f1a0>)),
                 ('zscore1', ZScore()),
                 ('bttda',
                  BTTDACV(clf=Pipeline(steps=[('functiontransformer',
                                               FunctionTransformer(func=<function NumpyBackend.to_numpy at 0x14830690f060>)),
                                              ('pca', PCA(whiten=T...
                          max_n_blocks=1, n_jobs=-1,
                          thetas=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8,
                                  0.9, 1.0])),
                 ('clf',
                  Pipeline(steps=[('functiontransformer',
                                   FunctionTransformer(func=<function NumpyBackend.to_numpy at 0x14830690f060>)),
                                  ('pca', PC

In [5]:
import joblib
from joblib import Parallel, delayed
import distributed
from IPython import display
import pandas as pd
from hpc import create_cluster, create_client, TIMEOUT

with create_cluster(cluster='cpu') as cluster, create_client(cluster) as client:
    results = []
    for dataset in datasets:
        print(f'Benchmarking on dataset {dataset.code}...')
        job_args = []
        for subject in dataset.subject_list:
            for pipe in pipelines.keys():
                job_args.append((dataset, subject,pipe))    
        with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
            results += Parallel(n_jobs=-1, verbose=True)(delayed(eval_moabb_within_session)(*args) for args in job_args)
results = pd.concat(results, ignore_index=True)

Benchmarking on dataset AlexandreMotorImagery...


[Parallel(n_jobs=-1)]: Using backend DaskDistributedBackend with 9 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 out of   8 | elapsed:  5.2min finished


In [6]:
results.to_csv('results/moabb_mi.csv')
results

,score,time,samples,subject,session,channels,n_sessions,dataset,pipeline
0,0.300000,31.453259,60.0,1,0,16,1,AlexandreMotorImagery,HODA
1,0.516667,25.735838,60.0,2,0,16,1,AlexandreMotorImagery,HODA
2,0.366667,35.824471,60.0,3,0,16,1,AlexandreMotorImagery,HODA
3,0.666667,26.239296,60.0,4,0,16,1,AlexandreMotorImagery,HODA
4,0.483333,49.166203,60.0,5,0,16,1,AlexandreMotorImagery,HODA
5,0.316667,49.245937,60.0,6,0,16,1,AlexandreMotorImagery,HODA
6,0.816667,25.871113,60.0,7,0,16,1,AlexandreMotorImagery,HODA
7,0.616667,26.063980,60.0,8,0,16,1,AlexandreMotorImagery,HODA


In [7]:
results = pd.read_csv('results/moabb_mi.csv')

In [8]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate(['mean', 'std'])

,,mean,std
dataset,pipeline,,
AlexandreMotorImagery,HODA,0.510417,0.182343


In [9]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean').reset_index().groupby('pipeline')['score'].aggregate('mean')

pipeline
HODA    0.510417
Name: score, dtype: float64

In [10]:
df_diff = results.pivot(index=['subject', 'session', 'channels', 'n_sessions', 'samples', 'dataset'], columns='pipeline', values='score')
df_diff = df_diff.reset_index()
df_diff['score_diff'] = df_diff['BTTDA'] - df_diff['HODA']
df_diff

KeyError: 'BTTDA'

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

def compare_score_plot(df, pipe1, pipe2):
    fig = px.scatter(df, x=pipe1, y=pipe2, color='dataset', facet_col='dataset', facet_col_wrap=5)
    fig.update_yaxes(scaleanchor="x")
    fig.update_xaxes(range=[0, 1])
    fig.update_yaxes(range=[0, 1])
    fig.add_shape(
        type="line",
        x0=0, y0=0.0, x1=1, y1=1,
        line=dict(color="gray", dash='dash'),
        layer="below" ,
        row='all', col='all', exclude_empty_subplots=True
    )
    

    return fig

fig = compare_score_plot(df_diff, 'HODA', 'BTTDA')
fig.update_layout(
    autosize=False,
    width=1800,
    height=1800,
)
fig.update_layout(showlegend=False)
fig